# Module 1: Data Analysis and Preprocessing
This notebook performs initial dataset analysis, text cleaning, and various tokenization methods (including BPE and morphological via `stanza`).

In [ ]:
!pip install -q stanza nltk scikit-learn transformers sentencepiece matplotlib seaborn

In [ ]:
import json
import re
import string
import stanza
import nltk
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

nltk.download('punkt')
nltk.download('punkt_tab')
# stanza.download('kk')
# nlp = stanza.Pipeline('kk')

from transformers import AutoTokenizer
bpe_tokenizer = AutoTokenizer.from_pretrained('google/mt5-small')

## 1. Load Data and Basic Statistics

In [ ]:
with open('abai-sozderi.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

df = pd.DataFrame(data)

df['char_length'] = df['text'].apply(len)
df['word_count'] = df['text'].apply(lambda x: len(x.split()))

print(f"Total documents (Words of Wisdom): {len(df)}")
print(f"Total tokens (approximate via spaces): {df['word_count'].sum()}")
print(f"Average text length (in words): {df['word_count'].mean():.2f}")

# Visualize length distribution
plt.figure(figsize=(10, 5))
sns.histplot(df['word_count'], bins=20, kde=True)
plt.title('Distribution of Text Lengths (in words)')
plt.xlabel('Number of words')
plt.ylabel('Frequency')
plt.show()

## 2. Train / Validation / Test Split

In [ ]:
# Split using 80/10/10 ratio
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Train set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")

train_df.to_csv('data/train.csv', index=False)
val_df.to_csv('data/val.csv', index=False)
test_df.to_csv('data/test.csv', index=False)

## 3. Preprocessing Modes
Functions for three different processing strategies.

In [ ]:
def preprocess_basic(text):
    # 1. Punctuation removal + lowercase + tokenization
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation + '«»—–'))
    tokens = word_tokenize(text)
    return tokens

def preprocess_morphological(text):
    # 2. Morphological tokenization + lemmatization (Stanza)
    doc = nlp(text)
    tokens = []
    for sent in doc.sentences:
        for word in sent.words:
            if word.lemma and word.lemma.isalpha():
                tokens.append(word.lemma.lower())
    return tokens

def preprocess_bpe(text):
    # 3. SentencePiece/BPE tokenizer
    return bpe_tokenizer.tokenize(text)

In [ ]:
sample_text = df['text'].iloc[0]
print("Original:\n", sample_text[:100], "...")
print("\n--- Option 1: Basic ---\n", preprocess_basic(sample_text[:100]))
# print("\n--- Option 2: Morphological ---\n", preprocess_morphological(sample_text[:100]))
print("\n--- Option 3: BPE ---\n", preprocess_bpe(sample_text[:100]))

## 4. Frequency Analysis and Topic Modeling

In [ ]:
from nltk.util import ngrams
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

all_basic_tokens = []
for t in df['text']:
    all_basic_tokens.extend(preprocess_basic(t))
    
# Word frequencies
word_freq = Counter(all_basic_tokens)
print("Most frequent words:\n", word_freq.most_common(10))

# Bigrams
bigram_freq = Counter(ngrams(all_basic_tokens, 2))
print("\nMost frequent bigrams:\n", bigram_freq.most_common(10))

# Topic Modeling (LDA)
corpus = [' '.join(preprocess_basic(text)) for text in df['text']]
vectorizer = CountVectorizer(max_df=0.9, min_df=2)
X = vectorizer.fit_transform(corpus)

lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(X)

terms = vectorizer.get_feature_names_out()
print("\nTopics (Topic Modeling):")
for idx, topic in enumerate(lda.components_):
    print(f"Topic {idx+1}: ", ', '.join([terms[i] for i in topic.argsort()[:-10-1:-1]]))